### 원하는 비율 조합 파일 생성 
- total reads: 5000 reads
- GBM percent: 0.1%, 0.5%, 1%, 2%, 2.5%, 3%, 5%, 10%, 100%

In [ ]:
import os
import random
import pysam

# 설정 값
total_reads = 5000  # 총 리드 수
mut_ratios = [0, 0.001, 0.005, 0.01, 0.02, 0.025, 0.03, 0.05, 0.1, 1]  # mut 비율 리스트 (0%, 0.1% 등)
num_files = 1000  # 생성할 파일 수

# 입력 BAM 파일 경로
mut_bam_file_path = "/sample_data/step03_Preprocessing for ML_RawData Mixing/mut_sorted.bam" 
wt_bam_file_path = "/sample_data/step03_Preprocessing for ML_RawData Mixing/wt_sorted.bam"

# 기본 출력 경로
base_output_dir = "/result/step03_Preprocessing for ML_RawData Mixing/"

# BAM 파일 열기
mut_bam = pysam.AlignmentFile(mut_bam_file_path, "rb")
wt_bam = pysam.AlignmentFile(wt_bam_file_path, "rb")

# Mut와 WT 리드 읽기
mut_reads = list(mut_bam.fetch())
wt_reads = list(wt_bam.fetch())

# header에서 reference_names와 reference_lengths 추출 (파일을 닫기 전에)
reference_names = mut_bam.references
reference_lengths = mut_bam.lengths

# BAM 파일 닫기
mut_bam.close()
wt_bam.close()

# R1, R2 쌍을 매칭하는 함수
def pair_reads(reads):
    paired_reads = {}
    for read in reads:
        if read.is_paired:
            pair_key = read.query_name
            if pair_key not in paired_reads:
                paired_reads[pair_key] = [None, None]  # [R1, R2]
            if read.is_read1:
                paired_reads[pair_key][0] = read
            elif read.is_read2:
                paired_reads[pair_key][1] = read
    # R1, R2 쌍이 모두 존재하는 경우만 반환
    return [pair for pair in paired_reads.values() if None not in pair]

# 비율별로 처리
for mut_ratio in mut_ratios:
    # 출력 디렉토리 생성 (mut 퍼센트별 폴더)
    folder_name = f"mut_{int(mut_ratio * 5000)}_reads"
    output_dir = os.path.join(base_output_dir, folder_name)
    os.makedirs(output_dir, exist_ok=True)  # 폴더 생성
    # 파일 수 만큼 반복
    for i in range(num_files):
        # Mut에서 샘플링할 쌍 수
        mut_pair_count = int(total_reads * mut_ratio) // 2 if mut_ratio > 0.0 else 0 
        # WT에서 샘플링할 쌍 수 (Mut를 제외한 나머지 비율)
        wt_pair_count = (total_reads // 2) - mut_pair_count
        # R1, R2 쌍 샘플링
        sampled_mut_pairs = pair_reads(mut_reads)
        sampled_wt_pairs = pair_reads(wt_reads)
        # Mut와 WT에서 각각 필요한 수만큼 R1, R2 쌍을 샘플링
        sampled_mut_pairs = random.sample(sampled_mut_pairs, mut_pair_count) if mut_pair_count > 0 else []
        sampled_wt_pairs = random.sample(sampled_wt_pairs, wt_pair_count) if wt_pair_count > 0 else []
        # 최종 샘플링된 리드 합치기 (R1, R2 순서대로 포함)
        final_sampled_reads = [read for pair in sampled_mut_pairs + sampled_wt_pairs for read in pair]
        # 출력 파일 경로
        output_bam_file_path = os.path.join(output_dir, f"sampled_reads_{i+1}.bam")
        block_count_output_path = os.path.join(output_dir, f"block_read_counts_{i+1}.txt")
        # 블록별 리드 수를 계산할 리스트
        block_read_counts = []
        # 블록별 리드 수 계산
        mut_block_counts = {}
        wt_block_counts = {}
        for read in final_sampled_reads:
            block_key = (read.reference_name, read.reference_start // 100 * 100)
            if read in [r for pair in sampled_mut_pairs for r in pair]:
                mut_block_counts[block_key] = mut_block_counts.get(block_key, 0) + 1
            else:
                wt_block_counts[block_key] = wt_block_counts.get(block_key, 0) + 1
        # 블록별 리드 수 저장
        all_blocks = set(mut_block_counts.keys()).union(wt_block_counts.keys())
        for block_key in sorted(all_blocks):
            chr_name, start_range = block_key
            end_range = start_range + 100
            mut_count = mut_block_counts.get(block_key, 0)
            wt_count = wt_block_counts.get(block_key, 0)
            block_read_counts.append({
                'chr': chr_name,
                'start_range': start_range,
                'end_range': end_range,
                'mut_reads': mut_count,
                'wt_reads': wt_count,
                'total_reads': mut_count + wt_count
            })
        # 블록별 리드 수를 텍스트 파일로 저장
        with open(block_count_output_path, "w") as count_file:
            count_file.write("chr\tstart_range\tend_range\tmut_reads\twt_reads\total_reads\n")
            for block in block_read_counts:
                count_file.write(
                    f"{block['chr']}\t{block['start_range']}\t{block['end_range']}\t"
                    f"{block['mut_reads']}\t{block['wt_reads']}\t{block['total_reads']}\n"
                )
        # 출력 BAM 파일 헤더 정보 제공 (reference_names, reference_lengths 추가)
        header = {
            'HD': {'VN': '1.0', 'SO': 'coordinate'},
            'SQ': [{'SN': name, 'LN': length} for name, length in zip(reference_names, reference_lengths)]
        }
        with pysam.AlignmentFile(output_bam_file_path, "wb", header=header) as out_bam:
            for read in final_sampled_reads:
                out_bam.write(read)
        # 결과 출력
        print(f"Mut ratio: {int(mut_ratio * 100)}%, WT ratio: {int((1 - mut_ratio) * 100)}%")
        print(f"Mut sampled pairs: {len(sampled_mut_pairs)}")
        print(f"WT sampled pairs: {len(sampled_wt_pairs)}")
        print(f"Sampled reads saved to {output_bam_file_path}")
        print(f"Block read counts saved to {block_count_output_path}")

ModuleNotFoundError: No module named 'pysam'